## Overview
This notebook implements a complete SAR (Synthetic Aperture Radar) change detection pipeline using Sentinel-1 data. The workflow uses the **SAR_GSD** package to process multi-temporal SAR imagery and identify areas of persistent backscatter change, which may indicate ground deformation, construction, or land cover changes.

#### Package Structure

The analysis uses modular Python code organized in the `SAR_GSD/` package:

- **`config.py`**: Configuration management (paths, parameters, credentials)
- **`download.py`**: Data acquisition from Sentinel Hub API
- **`processing.py`**: Time series analysis and change detection
- **`visualization.py`**: Plotting and figure generation
- **`export.py`**: KML and GeoTIFF export functions

#### Configuration

All configuration is managed through the `Config` class in `SAR_GSD/config.py`:

```python
from SAR_GSD import Config

# Access configuration
print(Config.OUTPUT_DIR)
print(Config.CHANGE_THRESHOLD)

# Get predefined study area
area = Config.get_study_area('ipatinga_test')
print(area['bbox'])

# Validate credentials
if Config.validate_sentinel_hub_credentials():
    print("Credentials OK")
```

#### API Credentials

Credentials are stored in the `.env` file:

```
SENTINEL_HUB_CLIENT_ID=your_client_id
SENTINEL_HUB_CLIENT_SECRET=your_client_secret
```

Get your free Sentinel Hub credentials at: https://apps.sentinel-hub.com/dashboard/

### 1 - IMPORTS AND CONFIGURATIONS

Configure the analysis parameters:
- Area of Interest (AOI) bounding box
- Time range for data acquisition
- Spatial resolution
- Change detection threshold

In [6]:
import os

# Import the SAR_GSD package
from SAR_GSD import *

print(f"Output directory: {Config.OUTPUT_DIR}")

# Define Area of Interest (AOI) as [lon_min, lat_min, lon_max, lat_max]
AOI_BBOX = Config.DEFAULT_AOI_BBOX
print(f"AOI Bounding Box: {AOI_BBOX}")

# Temporal parameters
START_DATE = Config.DEFAULT_START_DATE
END_DATE = None  # None = use today's date
MAX_DATES = Config.MAX_DATES  # Maximum number of acquisitions to process
print(f"Start date: {START_DATE}")
print(f"Maximum dates to process: {MAX_DATES}")

# Spatial resolution in meters
RESOLUTION = Config.DEFAULT_RESOLUTION
print(f"Resolution: {RESOLUTION} m")

# Change detection threshold (log-intensity change per year)
CHANGE_THRESHOLD = Config.CHANGE_THRESHOLD
print(f"Change threshold: ±{CHANGE_THRESHOLD} log-units/year")

# Polarization
POLARIZATION = Config.POLARIZATION
print(f"Polarization: {POLARIZATION}")

print("\nConfiguration loaded from Config class.")

Output directory: /content/drive/MyDrive/SciProgr2025_Assign2_Avinash_Flavio/outputs
AOI Bounding Box: [-42.85, -19.65, -42.48, -19.4]
Start date: 2017-01-01
Maximum dates to process: 40
Resolution: 20 m
Change threshold: ±0.07 log-units/year
Polarization: VV

Configuration loaded from Config class.


### 2 - BUILD DATACUBE

- Queries Sentinel Hub for available Sentinel-1 IW GRD acquisitions
- Downloads VV polarization backscatter data for the specified AOI and time period
- Normalizes each image by its median value to reduce radiometric variations
- Builds a 3D xarray DataArray with dimensions (time, y, x)
- Includes proper geospatial coordinates and CRS (EPSG:4326)
- Saves the datacube as NetCDF format

In [ ]:
# Build the datacube using the download module
datacube, valid_dates = build_datacube(
    bbox_coords=AOI_BBOX,
    start_date=START_DATE,
    end_date=END_DATE,
    resolution=RESOLUTION,
    max_dates=MAX_DATES,
    polarization=POLARIZATION,
    normalize=True,  # Normalize by median
    verbose=True,
)

In [ ]:
# Save datacube
datacube.to_netcdf(Config.get_output_path(Config.DATACUBE_FILENAME))

### 3 - PROCESS TIME SERIES
- Applies linear regression to log-transformed backscatter time series
- Computes trend (slope) for each pixel in log-intensity units per year
- Applies spatial smoothing (Gaussian filter + mean filter) to reduce noise
- Identifies pixels with trends exceeding the threshold (±0.07 log-units/year)
- Generates binary masks for positive and negative changes
- Exports trend map and masks as GeoTIFF files

In [56]:
# If needed, now the datacube can be loaded
datacube_path = Config.get_output_path(Config.DATACUBE_FILENAME)
# datacube_path = r"D:\sar_datacube.nc"
datacube = xr.open_dataset(datacube_path)
print(f'Datacube loaded from {datacube_path}')
datacube

Datacube loaded from /content/drive/MyDrive/SciProgr2025_Assign2_Avinash_Flavio/outputs/sar_datacube.nc


<xarray.Dataset> Size: 490MB
Dimensions:      (time: 46, y: 1358, x: 1961)
Coordinates:
  * time         (time) datetime64[ns] 368B 2017-01-03 2017-03-16 ... 2025-11-29
  * y            (y) float64 11kB -19.4 -19.4 -19.4 ... -19.65 -19.65 -19.65
  * x            (x) float64 16kB -42.85 -42.85 -42.85 ... -42.48 -42.48 -42.48
Data variables:
    backscatter  (time, y, x) float32 490MB ...
    spatial_ref  int64 8B ...

In [57]:
# Check PyTorch and CUDA availability
import torch

# Check available hardware
device = get_device(verbose=True)

No GPU detected, using CPU


In [58]:
# Run the complete processing pipeline
results = process_sar_timeseries(
    datacube=datacube['backscatter'],
    threshold=CHANGE_THRESHOLD,
    gaussian_sigma=Config.GAUSSIAN_SIGMA,
    mean_filter_size=Config.MEAN_FILTER_SIZE,
    device = device,
    verbose=True,
)

# Extract results
trend = results["trend"]
pvalues = results["pvalues"]
positive_mask = results["positive_mask"]
negative_mask = results["negative_mask"]

SAR TIME SERIES PROCESSING PIPELINE

[1/3] Computing temporal trend...
Time span: 0.00 to 8.90 years
Number of acquisitions: 46
Trend range: -0.3934 to 0.4438 log-units/year
Significant trends (p < 0.05): 1455302/2662172 (54.7%)
Highly significant (p < 0.01): 1152912/2662172 (43.3%)
Temporal trend computed in 14.07 seconds

[2/3] Applying spatial smoothing...
Applying spatial smoothing...
  Gaussian filter: sigma=1.5 pixels
  Mean filter: 3x3 kernel
  Output range: -0.2171 to 0.2342
Spatial smoothing applied in 0.91 seconds

[3/3] Detecting changes...

Change Detection Results:
  Threshold: ±0.07 log-units/year
  Positive changes: 59153 pixels (2.22%)
  Negative changes: 65389 pixels (2.46%)
  Stable areas: 2538496 pixels (95.32%)
Change detection completed in 0.01 seconds


In [60]:
# Update datacube
datacube["trend"] = (["y", "x"], trend)
datacube["pvalues"] = (["y", "x"], pvalues)
datacube["positive_mask"] = (["y", "x"], positive_mask.astype(np.uint8))
datacube["negative_mask"] = (["y", "x"], negative_mask.astype(np.uint8))
datacube

<xarray.Dataset> Size: 527MB
Dimensions:        (time: 46, y: 1358, x: 1961)
Coordinates:
  * time           (time) datetime64[ns] 368B 2017-01-03 ... 2025-11-29
  * y              (y) float64 11kB -19.4 -19.4 -19.4 ... -19.65 -19.65 -19.65
  * x              (x) float64 16kB -42.85 -42.85 -42.85 ... -42.48 -42.48
Data variables:
    backscatter    (time, y, x) float32 490MB 0.4967 0.4462 ... 0.4232 0.9313
    spatial_ref    int64 8B ...
    trend          (y, x) float32 11MB -0.0006168 2.061e-05 ... -0.007731
    pvalues        (y, x) float64 21MB 0.05246 0.363 ... 0.7187 7.358e-05
    positive_mask  (y, x) uint8 3MB 0 0 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0 0 0 0
    negative_mask  (y, x) uint8 3MB 0 0 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0 0 0 0

### 4 - EXPORT TO KML AND EXPORT OUTPUTS

- Vectorizes change masks into polygons
- Creates KML file with color-coded polygons for Google Earth
- Blue polygons: positive backscatter trend (increasing intensity)
- Red polygons: negative backscatter trend (decreasing intensity)

In [33]:
# Save all outputs (this includes KML creation)
output_files = save_all_outputs(
    datacube["trend"],
    datacube["pvalues"],
    positive_mask=datacube["positive_mask"],
    negative_mask=datacube["negative_mask"],
    bbox=AOI_BBOX,
    output_dir=Config.OUTPUT_DIR,
    verbose=True,
)

print("\nAll output files saved!")
print("\nOutput files:")
for key, path in output_files.items():
    print(f"  {key}: {path}")


Saving trend map (GeoTIFF)...
  ✓ /content/drive/MyDrive/SciProgr2025_Assign2_Avinash_Flavio/outputs/sar_trend.tif

Saving pvalues map (GeoTIFF)...
  ✓ /content/drive/MyDrive/SciProgr2025_Assign2_Avinash_Flavio/outputs/sar_pvalues.tif

Saving positive mask (GeoTIFF)...
  ✓ /content/drive/MyDrive/SciProgr2025_Assign2_Avinash_Flavio/outputs/positive_change_mask.tif

Saving negative mask (GeoTIFF)...
  ✓ /content/drive/MyDrive/SciProgr2025_Assign2_Avinash_Flavio/outputs/negative_change_mask.tif

All output files saved!

Output files:
  trend: /content/drive/MyDrive/SciProgr2025_Assign2_Avinash_Flavio/outputs/sar_trend.tif
  pvalues: /content/drive/MyDrive/SciProgr2025_Assign2_Avinash_Flavio/outputs/sar_pvalues.tif
  positive_mask: /content/drive/MyDrive/SciProgr2025_Assign2_Avinash_Flavio/outputs/positive_change_mask.tif
  negative_mask: /content/drive/MyDrive/SciProgr2025_Assign2_Avinash_Flavio/outputs/negative_change_mask.tif


In [34]:
# Create summary report
report_path = create_summary_report(
    outputs=output_files,
    datacube=datacube,
    trend_da=datacube['trend'],
    positive_mask=datacube['positive_mask'],
    negative_mask=datacube['negative_mask'],
)

# Display the report
with open(report_path, "r") as f:
    print(f.read())


Summary report saved: /content/drive/MyDrive/SciProgr2025_Assign2_Avinash_Flavio/outputs/summary.txt

    SAR GROUND SURFACE CHANGE DETECTION - SUMMARY REPORT

    DATA ACQUISITION
    ----------------------------------------------------------------------
    Source: Sentinel-1 VV polarization
    Date range: 2017-01-03 to 2025-11-29
    Number of acquisitions: 46
    Spatial resolution: N/A meters

    TREND ANALYSIS
    ----------------------------------------------------------------------
    Trend range: -0.2171 to 0.2342 log-units/year
    Mean trend: 0.0001 log-units/year
    Method: Linear regression

    CHANGE DETECTION
    ----------------------------------------------------------------------
    Threshold: ±0.07 log-units/year
    Total pixels: 2,663,038
    Positive changes: 59,153 pixels (2.22%)
    Negative changes: 65,389 pixels (2.46%)
    Stable areas: 2,538,496 pixels

    OUTPUT FILES
    ----------------------------------------------------------------------
      t

### 5 - DOWNLOAD DEM DATA, PROCESS SLOPE AND CREATE SLOPE CLASSES

In [ ]:
# Download and save DEM from Open Topography
dem = get_dem(AOI_BBOX)

Attempting to load DEM from OpenTopography.
CRS: GEOGCS["WGS 84",DATUM["World Geodetic System 1984",SPHEROID["WGS 84",6378137,298.257223563]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AXIS["Latitude",NORTH],AXIS["Longitude",EAST]]
Data range: 212.00m to 1240.00m
Saving DEM to d:\Flávio Rocha USER\OneDrive\GIT\SAR_GSD\outputs\dem.tif


#### 5.1 - Add DEM to the Datacube, and reproject the cube to an adequate CRS in meters

In [68]:
import rioxarray
import xarray as xr

#### Add DEM data to the datacube
#Load DEM if needed
dem = rioxarray.open_rasterio(Config.get_output_path(Config.DEM_FILENAME), masked=True)

# Set datacube CRS
datacube.rio.write_crs("EPSG:4326", inplace=True)

# Reproject DEM to match datacube's CRS and resolution
dem_reprojected = dem.rio.reproject_match(datacube)

# Add DEM to datacube
datacube["dem"] = (["y", "x"], dem_reprojected.squeeze().to_numpy())
datacube

# Reproject the dem to a appropriate SRC (in meters) for the analysis
# For this area, we will use EPSG:31983 (SIRGAS 2000 23S)
datacube = datacube.rio.reproject("EPSG:31983")

#######################################
# If wanted, save another version of the datacube with the DEM
datacube.to_netcdf(Config.get_output_path("sar_datacube_reproj.nc"))

#### 5.2 - Compute slope and create slope classes

In [ ]:
# # If needed, reload the datacube already reproject to EPSG:31983
# datacube = xr.open_dataset(Config.get_output_path("sar_datacube_reproj_2401.nc"))
# datacube = datacube.rio.write_crs("EPSG:31983")
# datacube = datacube.rio.set_spatial_dims(x_dim="x", y_dim="y")

In [73]:
# Compute slope and create Slope classes
slope = calculate_slope(datacube["dem"])
datacube["slope"] = (["y", "x"], slope.to_numpy())

# Create classes of slope
slope_classes = {1: "0-5°", 2: "5-10°", 3: "10-20°", 4: "20-45°", 5: "45-75°", 6: "75-90°"}

#Update the datacube with slope classes
datacube["slope_class"] = (["y", "x"], np.digitize(datacube["slope"].to_numpy(), bins=[0, 5, 10, 20, 45, 75]))
datacube["slope_class"].attrs = {'Slope classes': slope_classes}



### 6 - CREATE VISUALIZATIONS

Generate figures showing:
- SAR intensity image (latest acquisition in log scale)
- Change detection overlay (intensity + colored change masks)
- Trend map with diverging colormap, and corresponding p-values for the linear regression.
- Saves all figures as PNG files in `outputs/`

In [76]:
# Generate all standard figures
figures = create_all_figures(
    datacube=datacube['backscatter'],
    trend=datacube['trend'],
    pvalues=datacube['pvalues'],
    positive_mask=positive_mask,
    negative_mask=negative_mask,
    threshold=CHANGE_THRESHOLD,
    output_dir=str(Config.OUTPUT_DIR),
    dpi=Config.DPI,
    show=True,  # Display figures in notebook
)

print("\nAll figures generated and saved!")

Output hidden; open in https://colab.research.google.com to view.

- **<font color='blue'>Blue areas</font>** (positive trend): Increasing backscatter (construction, vegetation growth, surface roughening)
- **<font color='red'>Red areas</font>** (negative trend): Decreasing backscatter (subsidence, deforestation, surface smoothing)

**<font color='orange'>Note</font>**: This pipeline uses log-ratio change detection with linear regression. It is sensitive to persistent temporal trends but less sensitive to rapid changes or seasonal variations. For different applications, consider adjusting the threshold, smoothing parameters, or analysis method.

### 7 - Raster-Vector Operations

- Using Fiona, read a geopackage file, iterate through geometries, create a buffer for them,
and save to a new geopackage file.
- Perform zonal statistics between target polygons and SAR change detection (positive and negative masks) results.
- Perform zonal statistics between the polygon masks and the trend of change raster.
- Perform zonal statistics between the polygon masks and Slope Classes

In [77]:
from shapely.geometry import box
from shapely.ops import transform
from pyproj import Transformer

# Import transmission lines power grid
# Feature set available at https://gisepeprd2.epe.gov.br/webmapepe/
path_tl = os.path.join(Config.DATA_DIR, 'TransmissionGrid.gpkg')

# Inspect available layers (very useful for GPKGs)
with fiona.open(path_tl) as src:
    print(f'Properties: {src.schema}')
    print(f'CRS:{src.crs}')

layers = fiona.listlayers(path_tl)
print(f'Layers:{layers}')
# Since the geopackage has only one layer, we can use the default layer name

# Using shapely, create a geometry out of the bounding box of the AOI
aoi = box(*AOI_BBOX)
# Create transformer from EPSG:4326 (WGS84) to EPSG:31983 (SIRGAS 2000 / UTM zone 23S)
transformer = Transformer.from_crs("EPSG:4326", "EPSG:31983", always_xy=True)
# Transform the geometry
aoi_projected = transform(transformer.transform, aoi)

# Vector operation 1 and 2: buffering and clipping
# Using Fiona, read a geopackage file, iterate through geometries, and perform 2 spatial/geometric at once
path_buff = Config.get_output_path('TransmissionGrid_50m_buffer.gpkg')
path_buff = buff_and_clip_geopackage(path_tl, path_buff,  buffer_distance=50, clip_geom=aoi_projected)


Properties: {'properties': {'Nome': 'str:254', 'Concession': 'str:254', 'Tensao': 'float', 'Extensao': 'float', 'Ano_Opera': 'int32', 'created_us': 'str:254', 'created_da': 'date', 'last_edite': 'str:254', 'last_edi_1': 'date', 'Shape_STLe': 'float'}, 'geometry': 'LineString'}
CRS:EPSG:31983
Layers:['TransmissionGrid']
Buffered geopackage saved to: /content/drive/MyDrive/SciProgr2025_Assign2_Avinash_Flavio/outputs/TransmissionGrid_50m_buffer.gpkg


In [78]:
# Vector operations 3: rasterization and zonal statistics
gdf_buff = gpd.read_file(path_buff)

# Pixel resolution
res = datacube.rio.resolution()[0]
path_zs = os.path.join(Config.OUTPUT_DIR, 'TransmissionGrid_zs.gpkg')

# Zonal statistics to compute the area of SAR intensity change within each Transmission Line buffer
gdf_tl = calculate_change_areas(
                                gdf_buff,
                                datacube["positive_mask"],
                                datacube["negative_mask"],
                                res,
                                output_path=path_zs
                                )

gdf_tl[:1]


Change areas saved to: /content/drive/MyDrive/SciProgr2025_Assign2_Avinash_Flavio/outputs/TransmissionGrid_zs.gpkg


,Nome,Concession,Tensao,Extensao,Ano_Opera,created_us,created_da,last_edite,last_edi_1,Shape_STLe,geometry,count_pos,pos_area_ha,count_neg,neg_area_ha,total_area_ha,pos_area_perc,neg_area_perc
0,"LT 230 kV Mesquita - Timóteo 2, C1",TIMOTEO MESQUITA - EMPRESA DE TRANSMISSAO TIMO...,230.0,23.812206,2022,GISUSER,2025-12-29,GISUSER,2025-12-29,0.218773,"POLYGON ((757694.234 7851081.42, 758165.438 78...",7.0,0.280288,181.0,7.247435,238.881838,0.117333,3.0339


In [80]:
# Call vectorize_and_analyze_changes(), which encapsulates a vectorization of positive/negative
# and creation of their GeoDataFrames, and zonal statistics of these geometries against
# the temporal-trend data and the slope classes
gdf_result = vectorize_and_analyze_changes(datacube)
gdf_result[:1]

Vectorizing positive change mask...
  Found 1613 polygons
Vectorizing negative change mask...
  Found 1781 polygons
Computing zonal statistics against trend raster...
Computing zonal statistics for 6 slope classes...


/usr/local/lib/python3.12/dist-packages/rasterstats/io.py:335: NodataWarning: Setting nodata to -999; specify nodata explicitly
  warnings.warn(


Created GeoDataFrame with 1613 positive and 1781 negative change polygons


,geometry,change_type,total_area_ha,mean_trend,count_0-5°,area_0-5°,area_0-5°_perc,count_5-10°,area_5-10°,area_5-10°_perc,...,area_10-20°_perc,count_20-45°,area_20-45°,area_20-45°_perc,count_45-75°,area_45-75°,area_45-75°_perc,count_75-90°,area_75-90°,area_75-90°_perc
0,"POLYGON ((726505.882 7853495.215, 726505.882 7...",positive,0.080082,0.070285,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,2.0,0.080082,100.0,0.0,0.0,0.0,0.0,0.0,0.0


### 8 - SUMMARY

#### 8.1 - Interactive Visualization

In [ ]:
import branca.colormap as cm
import matplotlib.colors as mcolors
import numpy as np
import folium

# Check intersection between transmission line buffers and SAR changes
# Cast to int to avoid issues with boolean values
gdf_result['intersects'] = gdf_result.geometry.intersects(gdf_tl.unary_union).astype(int)

# Get min/max for both variables
min_val, max_val = gdf_result["mean_trend"].min(), gdf_result["mean_trend"].max()

# Make the scale symmetric around zero
abs_max = max(abs(min_val), abs(max_val))

# Build a colormap centered at 0
sart_cmap = cm.linear.Spectral_08.scale(-abs_max, abs_max)
sart_cmap.caption = "SAR temporal trend (log-units/year)"

def sar_changes_stylefunc(feature):
    intersect = feature["properties"]["intersects"]
    color = feature["properties"]["mean_trend"]

    return {
        "fillColor": sart_cmap(color),
        "color": "black" if not intersect else "red",
        "weight": (0.3 + abs(color) * 2) if not intersect else (0.3 + abs(color) * 2)*5,
    }

geojson_display_dicts = [
{
    'data': gdf_tl[["Nome", "Concession", "pos_area_ha", "neg_area_ha", "total_area_ha", "pos_area_perc", "neg_area_perc", "geometry"]],
    'name': 'Transmission Lines',
    'attribute_map':
    {
        "Nome": "Name",
        "pos_area_ha": "Positive Area (ha)",
        "neg_area_ha": "Negative Area (ha)",
        "total_area_ha": "Total Area (ha)",
        "pos_area_perc": "Positive Area (%)",
        "neg_area_perc": "Negative Area (%)",
    },
    'feature_settings':
    {
        "color": "blue",
        "weight": 0.5,
        "fillOpacity": 0.5,
    },
    'highlight_function': lambda feature:
    {
        "weight": 2,
        "color": "#ff7800"
    }
},
{
    'data': gdf_result[
        ["change_type", "mean_trend", "total_area_ha",
        "area_0-5°_perc", "area_5-10°_perc", "area_10-20°_perc", "area_20-45°_perc",
        'area_45-75°_perc', "area_75-90°_perc", "intersects", "geometry"]],
    'name': 'SAR Changes',
    'attribute_map':
    {
        "change_type": "Change Type",
        "mean_trend": "Mean temporal trend (log-units/year)",
        "total_area_ha": "Total Area (ha)",
        "area_0-5°_perc": "Area 0-5° (%)",
        "area_5-10°_perc": "Area 5-10° (%)",
        "area_10-20°_perc": "Area 10-20° (%)",
        "area_20-45°_perc": "Area 20-45° (%)",
        "area_45-75°_perc": "Area 45-75° (%)",
        "area_75-90°_perc": "Area 75-90° (%)",
    },
    'feature_settings': sar_changes_stylefunc,
    'highlight_function': lambda feature:
    {
    "weight": 2,
    "color": "#f1f1f1"
    }
}
]

fmap = display_Folium_map(geojson_display_dicts, zoom_start=12)

############################
# Get the first backscatter image
sar_data = datacube["backscatter"][0]
first_date = str(sar_data.time.values).split('T')[0]

# Add to existing folium map
add_raster_to_folium(
    fmap,
    sar_data,
    name=f'SAR Intensity ({first_date})',
    opacity=0.6,
    cmap='gray',
    log_transform=True
)

############################
# Get the last backscatter image
sar_data = datacube["backscatter"][-1]
last_date = str(sar_data.time.values).split('T')[0]

# Add to existing folium map
add_raster_to_folium(
    fmap,
    sar_data,
    name=f'SAR Intensity ({last_date})',
    opacity=0.6,
    cmap='gray',
    log_transform=True
)

############################
# Display elevation
sar_data = datacube["dem"]

# Add to existing folium map
add_raster_to_folium(
    fmap,
    sar_data,
    name=f'Elevation (m)',
    opacity=0.6,
    cmap='gray',
)

############################
# Display slope classes
sar_data = datacube["slope_class"]

# Get slope class mapping
slope_classes = datacube["slope_class"].attrs['Slope classes']

# Create step colormap
colors = ['#f2f0f7', '#dadaeb', '#bcbddc', '#9e9ac8', '#807dba', '#6a51a3']
slope_cmap = cm.StepColormap(
    colors=colors,
    vmin=0.5,
    vmax=6.5,
    index=[1, 2, 3, 4, 5, 6, 7],  # Class boundaries
    caption='Slope Classes ()'
)

# Update the colormap caption with class labels
slope_cmap.caption = """Slope Classes\n1: '0-5°'\n2: '5-10°'\n3: '10-20°'\n4: '20-45°'\n5: '45-75°'\n6: '75-90°'"""

# Add to existing folium map
add_raster_to_folium(
    fmap,
    sar_data,
    name=f'Slope classes',
    opacity=0.6,
    cmap=mcolors.ListedColormap(colors),
    vmin=1,
    vmax=6
)

############################
# Add color maps and layer control to the map
fmap.add_child(sart_cmap)
fmap.add_child(slope_cmap)
folium.LayerControl().add_to(fmap)

############################
# Save if needed
fmap.save(os.path.join(Config.OUTPUT_DIR, "SARGSD_Example.html"))

# Display
fmap

Output hidden; open in https://colab.research.google.com to view.

### 9 - LAUNCH GOOGLE EARTH (OPTIONAL)

Attempt to automatically open the KML file in Google Earth Pro.
This is optional and will only work if Google Earth Pro is installed.

In [ ]:
# Convert transmission line buffers to kml
gdf_to_kml(
    gdf_tl,
    output_path=os.path.join(str(Config.OUTPUT_DIR),"transmission_line_buffers.kml"),
    name_column="Nome",
    description_columns=["Concession", "pos_area_ha", "neg_area_ha", "total_area_ha", "pos_area_perc", "neg_area_perc"],
    color="ff00ffff",      # AABBGGRR format
    fill_color="7fff0000",
    line_width=2.0,
    verbose=True
)

# Convert SAR mask to kml
gdf_to_kml(
    gdf_result[gdf_result["change_type"] == "positive"],
    output_path=os.path.join(str(Config.OUTPUT_DIR),"positive_change.kml"),
    name_column="mean_trend",
    description_columns=["total_area_ha","area_0-5°_perc", "area_5-10°_perc", "area_10-20°_perc", "area_20-45°_perc",'area_45-75°_perc', "area_75-90°_perc"],
    color="ff00ff00",      # Green (AABBGGRR format)
    fill_color="7f00ff00",
    line_width=0.8,
    verbose=True
)

# Convert SAR mask to kml
gdf_to_kml(
    gdf_result[gdf_result["change_type"] == "negative"],
    output_path=os.path.join(str(Config.OUTPUT_DIR),"negative_change.kml"),
    name_column="mean_trend",
    description_columns=["total_area_ha","area_0-5°", "area_5-10°", "area_10-20°", "area_20-45°","area_45-75°", "area_75-90°"],
    color="ff0000ff",      # Green (AABBGGRR format)
    fill_color="7f0000ff",
    line_width=0.8,
    verbose=True
)

# Launch google earth with multiple KML files
launch_google_earth([
    os.path.join(str(Config.OUTPUT_DIR),"transmission_line_buffers.kml"),
    os.path.join(str(Config.OUTPUT_DIR),"positive_change.kml"),
    os.path.join(str(Config.OUTPUT_DIR),"negative_change.kml")
])

Exporting GeoDataFrame to KML: d:\Flávio Rocha USER\OneDrive\GIT\SAR_GSD\outputs\transmission_line_buffers.kml
  Reprojecting from EPSG:31983 to EPSG:4326...
Exported 17 features to d:\Flávio Rocha USER\OneDrive\GIT\SAR_GSD\outputs\transmission_line_buffers.kml
Exporting GeoDataFrame to KML: d:\Flávio Rocha USER\OneDrive\GIT\SAR_GSD\outputs\positive_change.kml
  Reprojecting from PROJCS["SIRGAS 2000 / UTM zone 23S",GEOGCS["SIRGAS 2000",DATUM["Sistema de Referencia Geocentrico para las AmericaS 2000",SPHEROID["GRS 1980",6378137,298.257222101]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433],AUTHORITY["EPSG","4674"]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",0],PARAMETER["central_meridian",-45],PARAMETER["scale_factor",0.9996],PARAMETER["false_easting",500000],PARAMETER["false_northing",10000000],UNIT["metre",1],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","31983"]] to EPSG:4326...
Exported 1607 features to d:\Flávio Rocha USER\OneDrive\GIT\

True